# Week 2 · Lab 01
## Local API (`requests`) + SQL Extraction to pandas

> **AI Engineering Academy** · Gamut Technology Services

Data engineering for LLM pipelines starts with **getting data out** — from REST
APIs and from SQL databases — reliably and reproducibly. In this lab you'll extract
the *same* synthetic dataset two ways, then transform, join, and validate it into a
clean artifact an LLM pipeline could consume.

### ⚙️ No external internet required
The setup cell builds a synthetic **Cordwell Home & Hardware** SQLite database and
starts a small **FastAPI** server (`lab_api.py`) on `http://127.0.0.1:8000` that
serves it. The API and the SQL half read the **same** `cordwell.db`. See
`API_REFERENCE.md` and `cordwell_data_dictionary.md` for the endpoints and schema.

### Learning objectives
1. Call a local REST API with **query parameters**, parse JSON, and handle status codes robustly.
2. Implement **cursor pagination** and an **exponential-backoff** retry policy that respects `429` (via `Retry-After`) and `5xx`.
3. Extract from **SQLite** into pandas with **parameterized queries** and `pd.read_sql_query`, including **chunked reads**.
4. **Join, validate, and profile** the data, then persist it to **Parquet** and **JSONL** for downstream LLM use.

### Time budget — ~110 min
| Segment | Time |
|---|---|
| Setup (build DB + start API) | 8 min |
| **A.** HTTP API extraction with `requests` | 40 min |
| **B.** SQL → pandas (params, chunking, Parquet) | 35 min |
| **C.** Transform · join · validate → LLM-ready | 22 min |
| Wrap-up | 5 min |

### Files beside this notebook
- `build_cordwell_db.py` — the database generator (run for you in setup).
- `lab_api.py` — the local API (started for you in setup).
- `API_REFERENCE.md`, `cordwell_data_dictionary.md` — reference docs.


In [ ]:
%pip install -r requirements.txt

In [ ]:
# --- Setup: build the local database, start the local API ------------------
# You make NO external network calls in this lab. A generator script builds a
# synthetic SQLite database (Cordwell Home & Hardware), and a small FastAPI app
# serves that SAME database on localhost. One dataset, two access paths:
#   * Part A talks to the local REST API with `requests`
#   * Part B/C read the SQLite file directly with pandas
import os, time, json, sqlite3
from pathlib import Path
import requests
import pandas as pd

import build_cordwell_db          # the generator (shipped beside this notebook)
from lab_api import start_server   # the local API (shipped beside this notebook)

# 1) Build the database if it is not already present (seeded / reproducible).
DB_PATH = "cordwell.db"
if not Path(DB_PATH).exists():
    stats = build_cordwell_db.build(DB_PATH, n_orders=10_000, seed=2025)
    print("Built database:", stats)
else:
    print("Database already present:", DB_PATH)

# 2) Point the API at the database and start it on a background thread.
os.environ["CORDWELL_DB"] = DB_PATH
BASE_URL = "http://127.0.0.1:8000"
server, _thread = start_server(port=8000)
for _ in range(50):
    try:
        if requests.get(f"{BASE_URL}/health", timeout=1).status_code == 200:
            break
    except requests.exceptions.RequestException:
        time.sleep(0.1)
print("Local API ready:", requests.get(f"{BASE_URL}/health", timeout=2).json())

def check(label, predicate):
    """Soft self-check: prints PASS/FAIL, never raises."""
    try:
        ok = bool(predicate())
    except Exception as exc:
        ok = False
        label = f"{label}  (raised {type(exc).__name__}: {exc})"
    print(("PASS " if ok else "FAIL "), label)
    return ok

print("ready.")

## Part A — HTTP API extraction with `requests`

The API exposes the orders table with **keyset (cursor) pagination** and optional
filters. Endpoint:

```
GET /v1/orders?limit=<int>&cursor=<int>&region=<str>&channel=<str>
    -> {"data": [order, ...], "next_cursor": <int|null>, "count": <int>}
```

`next_cursor` is the last `order_id` of the page (pass it back as `cursor`), and it
is `null` on the final page.


### A1 — Warm-up: GET with params & `.json()`  *(guided)*

Always pass a **`timeout`**, and read the response envelope before assuming its shape.


In [ ]:
resp = requests.get(f"{BASE_URL}/v1/orders", params={"limit": 3}, timeout=10)
print("status:", resp.status_code)
payload = resp.json()
print("envelope keys:", list(payload.keys()))
print("count:", payload["count"], "| next_cursor:", payload["next_cursor"])
payload["data"][0]

### A2 — A robust request helper with retry + backoff

Real extraction code must survive transient failures. Implement
`get_json(url, params=None, *, max_retries=5)` that:

- issues a `GET` with a **timeout**;
- on **200**, returns the parsed JSON (`resp.json()`);
- on **429 / 500 / 502 / 503 / 504**, waits and retries with **exponential backoff**,
  and if the response carries a **`Retry-After`** header, waits *that many seconds*
  instead;
- on any **other 4xx**, raises `APIError` immediately (not retryable);
- on a `Timeout`/`ConnectionError`, waits and retries;
- after `max_retries`, raises `APIError`.

The API gives you two endpoints to prove it works: `/v1/unreliable` (503s a set number
of times, then 200) and `/v1/rate-limited` (429 + `Retry-After: 2`, then 200).


In [ ]:
class APIError(Exception):
    pass

def get_json(url, params=None, *, max_retries=5, timeout=15):
    backoff = 0.3
    last_exc = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.get(url, params=params, timeout=timeout)
        except (requests.Timeout, requests.ConnectionError) as exc:
            last_exc = exc
            time.sleep(backoff); backoff *= 2
            continue
        if resp.status_code == 200:
            return resp.json()
        if resp.status_code in (429, 500, 502, 503, 504):
            retry_after = resp.headers.get("Retry-After")
            wait = float(retry_after) if retry_after is not None else backoff
            time.sleep(wait); backoff *= 2
            continue
        raise APIError(f"Unexpected status {resp.status_code}: {resp.text[:200]}")
    raise APIError(f"Failed after {max_retries} attempts: {url} ({last_exc})")

requests.post(f"{BASE_URL}/admin/reset", timeout=5)
print("unreliable:", get_json(f"{BASE_URL}/v1/unreliable", {"key": "a2", "fail_times": 2}))
print("rate-limited:", get_json(f"{BASE_URL}/v1/rate-limited", {"key": "a2rl"}))

In [ ]:
# Recovers from transient 503s (2 failures -> success on attempt 3)
requests.post(f"{BASE_URL}/admin/reset", timeout=5)
_u = get_json(f"{BASE_URL}/v1/unreliable", {"key": "chkA2", "fail_times": 2})
check("A2: recovered from 503s via backoff (attempts == 3)", lambda: _u["attempts"] == 3)

# Honors Retry-After on a 429
requests.post(f"{BASE_URL}/admin/reset", timeout=5)
_r = get_json(f"{BASE_URL}/v1/rate-limited", {"key": "chkA2rl"})
check("A2: recovered from 429 (attempts == 2)", lambda: _r["attempts"] == 2)

# Non-retryable 4xx raises immediately
def _expect_apierror():
    try:
        get_json(f"{BASE_URL}/v1/does-not-exist")
        return False
    except APIError:
        return True
check("A2: non-retryable 4xx raises APIError", _expect_apierror)

🧑‍🏫 **Instructor note — A2.** Three ideas to land: (1) **retry only what's retryable** —
`5xx`/`429`/connection blips, never a `404`/`401` (retrying a bad request just wastes
time); (2) **respect `Retry-After`** — the server tells you how long to wait, so obey it
instead of guessing; (3) **exponential backoff** prevents hammering a struggling service.
Reset the counters before each demo or a re-run "passes" without actually retrying (the
server already counted past its threshold). In production you'd add a little random
**jitter** to the sleep to avoid a thundering herd, and often use `requests`' built-in
`Retry` adapter — mention both; the hand-rolled version here makes the logic explicit.


### A3 — Pagination: assemble the full orders table
Write `fetch_all_orders(base_url, page_size=1000, **filters)` that walks the cursor
pagination using your `get_json` and returns a single `pd.DataFrame` of every order.
Start with `cursor=0`; after each page, read `next_cursor` and pass it back as `cursor`;
stop when `next_cursor` is `None`. Forward any `filters` (like `region=...`) as params.


In [ ]:
def fetch_all_orders(base_url, page_size=1000, **filters):
    rows = []
    params = {"limit": page_size, "cursor": 0, **filters}
    url = f"{base_url}/v1/orders"
    while True:
        payload = get_json(url, params)
        rows.extend(payload["data"])
        if payload["next_cursor"] is None:
            break
        params["cursor"] = payload["next_cursor"]
    return pd.DataFrame(rows)

orders_df = fetch_all_orders(BASE_URL)
print("fetched orders:", orders_df.shape)
orders_df.head()

In [ ]:
check("A3: fetched all 10,000 orders", lambda: len(orders_df) == 10_000)
check("A3: order_ids are unique", lambda: orders_df["order_id"].is_unique)
check("A3: has the expected columns",
      lambda: {"order_id", "store_region", "channel", "order_date"} <= set(orders_df.columns))

🧑‍🏫 **Instructor note — A3.** Keyset (cursor) pagination is the real-world default for
large tables: instead of `OFFSET N` (which the DB must scan past), the server does
`WHERE order_id > :cursor ORDER BY order_id LIMIT :n` — O(1) per page and stable under
inserts. The stop signal here is `next_cursor is None`, *not* an empty page. Because
`fetch_all_orders` calls `get_json`, every page inherits the retry/backoff from A2 for
free — that layering (transport concern separate from pagination concern) is the point.


### A4 — Server-side filtering via query params
Fetching everything and filtering in pandas wastes bandwidth. The API filters
server-side: pass `region=` (and/or `channel=`). Fetch **only** the `Southeast` orders
into `southeast_df`, and cross-check the count against the database directly.


In [ ]:
southeast_df = fetch_all_orders(BASE_URL, region="Southeast")

_con = sqlite3.connect(DB_PATH)
sql_southeast = _con.execute(
    "SELECT COUNT(*) FROM orders WHERE store_region = ?", ("Southeast",)
).fetchone()[0]
_con.close()
print("API rows:", len(southeast_df), "| SQL count:", sql_southeast)

In [ ]:
check("A4: every returned order is in the Southeast region",
      lambda: (southeast_df["store_region"] == "Southeast").all())
check("A4: API filter count matches the database count",
      lambda: len(southeast_df) == sql_southeast)

🧑‍🏫 **Instructor note — A4.** Two lessons: **push filters to the source** (the server
returns only what you need — less data over the wire, less memory), and **the API is
just a view over the same SQL** — the `region=` param becomes a `WHERE store_region = ?`
on the server. The cross-check (API count == SQL count) is a great habit: when two paths
to the same data disagree, you have a bug. Note the SQL count uses a **parameterized**
query (`?`) — which is exactly Part B's topic.


## Part B — SQL → pandas with SQLite

Now read the database **directly**. `pd.read_sql_query` runs SQL and returns a
DataFrame; a raw `sqlite3` connection is all you need (no SQLAlchemy for SQLite).


### B1 — Parameterized queries (with a join)
Write SQL that joins `order_lines → orders → products` and returns line-level rows for
a given region **and** on/after a given date, then run it with **parameter binding**.
Fill `region` and `since` as Python values and pass them via `params=` — never with
f-strings or string concatenation.

Return columns: `order_id, order_date, store_region, channel, category, product_name,
quantity, unit_price, discount_pct`. Filter: `store_region = :region AND order_date >= :since`.


In [ ]:
conn = sqlite3.connect(DB_PATH)

region = "Southeast"
since = "2025-01-01"

sql = """
SELECT o.order_id, o.order_date, o.store_region, o.channel,
       p.category, p.product_name,
       ol.quantity, ol.unit_price, ol.discount_pct
FROM order_lines ol
JOIN orders   o ON ol.order_id = o.order_id
JOIN products p ON ol.product_id = p.product_id
WHERE o.store_region = ? AND o.order_date >= ?
"""
lines_df = pd.read_sql_query(sql, conn, params=(region, since))
print(lines_df.shape)
lines_df.head()

In [ ]:
check("B1: returned some rows", lambda: len(lines_df) > 0)
check("B1: filter held (region)", lambda: (lines_df["store_region"] == "Southeast").all())
check("B1: filter held (date)", lambda: (lines_df["order_date"] >= "2025-01-01").all())
check("B1: join populated product columns",
      lambda: lines_df["category"].notna().all() and lines_df["product_name"].notna().all())

🧑‍🏫 **Instructor note — B1.** The headline is **parameter binding**: `?` placeholders
let the driver bind values safely, which prevents **SQL injection** and handles quoting/
escaping for you. Show the anti-pattern once — `f"... WHERE store_region = '{region}'"` —
and note that a value like `x' OR '1'='1` would break it wide open. The join is standard
star-schema navigation: line facts out to the order and product dimensions. Since dates
are ISO `YYYY-MM-DD` strings, lexical `>=` comparison sorts correctly.


### B2 — Chunked reads → Parquet
The `order_lines` table is large (~45k rows here; imagine millions). Read it in
**chunks** so you never hold the whole table in memory, streaming each chunk to a single
Parquet file. Implement the streaming write, then confirm the row count round-trips.


In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

OUT_PARQUET = "order_lines.parquet"
chunks = pd.read_sql_query("SELECT * FROM order_lines", conn, chunksize=10_000)

writer = None
rows_written = 0
for chunk in chunks:
    table = pa.Table.from_pandas(chunk, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(OUT_PARQUET, table.schema)
    writer.write_table(table)
    rows_written += len(chunk)
if writer is not None:
    writer.close()
print("rows written:", rows_written)

In [ ]:
_reloaded = pd.read_parquet(OUT_PARQUET)
_con = sqlite3.connect(DB_PATH)
_sql_count = _con.execute("SELECT COUNT(*) FROM order_lines").fetchone()[0]
_con.close()
check("B2: parquet file was written", lambda: Path(OUT_PARQUET).exists())
check("B2: rows_written matches the table row count", lambda: rows_written == _sql_count)
check("B2: parquet round-trips the same row count", lambda: len(_reloaded) == _sql_count)

🧑‍🏫 **Instructor note — B2.** **Chunking** is the memory lever: `chunksize=` turns one
giant result set into a stream of DataFrames, so peak memory is one chunk, not the whole
table. The **trade-off** is more, smaller round-trips and slightly more code. The
`ParquetWriter` pattern is the streaming-write counterpart — open once (schema from the
first chunk), append each chunk, close once — producing a single columnar file. Parquet
is the right hand-off format: compressed, typed, and column-selectable downstream. In
pandas 3.0 you can also pass `dtype_backend="pyarrow"` to `read_sql_query` for Arrow-
backed dtypes end-to-end (a nice stretch).


### B3 — Validation snapshot & profiling
Before trusting extracted data, **profile** it and assert your expectations. Reload the
Parquet, build a small profile, and validate a few invariants.


In [ ]:
ol = pd.read_parquet(OUT_PARQUET)

profile = ol.describe()
null_counts = ol.isnull().sum()

quantity_ok = bool((ol["quantity"] >= 1).all())
price_ok = bool((ol["unit_price"] > 0).all())
discount_ok = bool(ol["discount_pct"].between(0, 100).all())
print(profile)
print("\nnulls per column:\n", null_counts)
print("invariants -> quantity:", quantity_ok, "price:", price_ok, "discount:", discount_ok)

In [ ]:
check("B3: no nulls in key columns",
      lambda: null_counts[["line_id", "order_id", "product_id", "quantity"]].sum() == 0)
check("B3: quantity >= 1 everywhere", lambda: quantity_ok is True)
check("B3: unit_price > 0 everywhere", lambda: price_ok is True)
check("B3: discount_pct within [0, 100]", lambda: discount_ok is True)

🧑‍🏫 **Instructor note — B3.** Profiling (`describe`, null counts, ranges) is how you
*earn* trust in extracted data before building on it — the deck's "profile first" habit,
now applied to a pipeline. Turning expectations into **assertions** is lightweight schema
validation; in a real pipeline these become a Pandera/Great Expectations contract that
fails the run when upstream data drifts. The point: validation is a **gate**, not a
one-time glance.


## Part C — Transform · join · validate → LLM-ready

Extraction is only step one. To feed an LLM workflow you usually **join** the pieces
into a denormalized view, **validate** it, and **serialize** it to a format the pipeline
consumes (here, JSONL — one JSON object per line, the lingua franca of LLM data).

> **Responsible-AI note.** Data quality *is* a safety control: an LLM fine-tuned or
> grounded on malformed, inconsistent data produces worse, less-trustworthy output.
> The validation below is the quality gate that protects everything downstream.


### C1 — Build and validate the enriched line-level view
Join `order_lines → orders → products` for **all** data, add a computed
`line_total = quantity * unit_price * (1 - discount_pct/100)` (rounded to 2 dp), and
validate referential + range invariants.


In [ ]:
sql_all = """
SELECT ol.line_id, o.order_id, o.store_region, o.channel, o.order_date,
       p.category, p.product_name,
       ol.quantity, ol.unit_price, ol.discount_pct
FROM order_lines ol
JOIN orders   o ON ol.order_id = o.order_id
JOIN products p ON ol.product_id = p.product_id
"""
enriched = pd.read_sql_query(sql_all, conn)
enriched["line_total"] = (
    enriched["quantity"] * enriched["unit_price"] * (1 - enriched["discount_pct"] / 100)
).round(2)

ref_ok = bool(enriched[["category", "product_name"]].notna().all().all())
total_ok = bool((enriched["line_total"] > 0).all())
print(enriched.shape)
enriched.head()

In [ ]:
_con = sqlite3.connect(DB_PATH)
_line_count = _con.execute("SELECT COUNT(*) FROM order_lines").fetchone()[0]
_con.close()
check("C1: every order line survived the joins (referential integrity)",
      lambda: len(enriched) == _line_count)
check("C1: all foreign keys resolved (no null product fields)", lambda: ref_ok is True)
check("C1: line_total computed and positive", lambda: total_ok is True)
check("C1: line_total math is correct on row 0",
      lambda: abs(enriched.loc[0, "line_total"]
                  - round(enriched.loc[0, "quantity"] * enriched.loc[0, "unit_price"]
                          * (1 - enriched.loc[0, "discount_pct"]/100), 2)) < 0.01)

🧑‍🏫 **Instructor note — C1.** An **inner join** that returns exactly the `order_lines`
row count proves **referential integrity** — no line lost a parent order or product. That
equality check is a cheap, powerful validation. `line_total` is a derived feature the raw
tables don't store; computing it here (once, validated) beats recomputing it ad hoc
downstream. If the join had returned *fewer* rows, you'd have orphan foreign keys to
investigate — exactly the kind of silent data bug validation is meant to catch.


### C2 — Serialize an LLM-ready JSONL artifact
LLM ingestion/fine-tuning pipelines consume **JSONL** (one JSON object per line).
Aggregate `enriched` to **one record per order** and write JSONL where each record has
structured fields plus a natural-language `text` summary.

Each record: `{"order_id", "store_region", "channel", "order_date", "n_lines",
"order_total", "text"}` where `text` is a sentence like
*"Order 42 (Southeast, Online) on 2025-03-04: 3 line items totaling $87.45."*


In [ ]:
grouped = (
    enriched.groupby("order_id")
    .agg(store_region=("store_region", "first"),
         channel=("channel", "first"),
         order_date=("order_date", "first"),
         n_lines=("line_id", "count"),
         order_total=("line_total", "sum"))
    .reset_index()
)
grouped["order_total"] = grouped["order_total"].round(2)
# grouped["text"] = grouped.apply(
#     lambda r: (f"Order {r['order_id']} ({r['store_region']}, {r['channel']}) "
#                f"on {r['order_date']}: {r['n_lines']} line item(s) "
#                f"totaling ${r['order_total']:.2f}."),
#     axis=1,
# )
grouped["text"] = (
    "Order " + grouped["order_id"].astype(str)
    + " (" + grouped["store_region"] + ", " + grouped["channel"] + ")"
    + " on " + grouped["order_date"].astype(str)
    + ": " + grouped["n_lines"].astype(str) + " line item(s)"
    + " totaling $" + grouped["order_total"].map("{:.2f}".format) + "."
)

OUT_JSONL = "orders_llm.jsonl"
cols = ["order_id", "store_region", "channel", "order_date", "n_lines", "order_total", "text"]
grouped[cols].to_json(OUT_JSONL, orient="records", lines=True)
n_written = sum(1 for _ in open(OUT_JSONL))
print("records written:", n_written)
print(grouped["text"].iloc[0])

In [ ]:
_records = [json.loads(line) for line in open(OUT_JSONL)]
check("C2: one JSONL record per order (10,000)", lambda: n_written == 10_000)
check("C2: JSONL is valid and parseable", lambda: len(_records) == 10_000)
check("C2: each record has the required fields",
      lambda: {"order_id", "store_region", "order_total", "text"} <= set(_records[0].keys()))
check("C2: the text summary is a non-empty string",
      lambda: isinstance(_records[0]["text"], str) and len(_records[0]["text"]) > 20)

🧑‍🏫 **Instructor note — C2.** **JSONL** is the default interchange format for LLM data
(fine-tuning files, batch inputs, eval sets) because it streams line-by-line without
loading the whole file. `to_json(orient="records", lines=True)` is the one-liner that
produces it. The `text` field models the real move of turning **structured rows into a
natural-language representation** an LLM can reason over, while keeping the structured
fields for filtering/metadata. Tie it back: everything upstream — reliable extraction,
parameterized SQL, validation — exists so this artifact is trustworthy.


## Wrap-up

You extracted the same dataset two ways (resilient REST calls and direct SQL),
transformed and validated it, and shipped Parquet + JSONL artifacts ready for an LLM
pipeline. Answer these in a markdown cell (they're the kind of thing a reviewer asks):

1. How does your retry/backoff behave for **429** vs **500** — what's the same, what's different?
2. Why are **parameterized** queries the default? Give a one-sentence example of an injection if they aren't used.
3. When would you choose **chunked** reads, and what trade-off do you accept?

**Stretch goals:** (a) add random **jitter** to the backoff sleep; (b) re-run B2 with
`dtype_backend="pyarrow"` and compare dtypes; (c) partition the JSONL by `store_region`.


In [ ]:
# Clean shutdown of the local API.
conn.close()
server.should_exit = True
time.sleep(0.3)
print("Local API stopped. Artifacts:", [p for p in ["order_lines.parquet", "orders_llm.jsonl"] if Path(p).exists()])